In [1]:
import re
import os
import requests
import html2text
from bs4 import BeautifulSoup
from collections import deque

pattern = r"https?:\/\/(?:www\d*\.)?eecs\.berkeley\.edu(?:\/[^\s]*)?"

os.makedirs("html", exist_ok=True)
os.makedirs("cleaned_text", exist_ok=True)

h2t = html2text.HTML2Text()
h2t.ignore_links = True
h2t.ignore_images = True
h2t.ignore_tables = False
h2t.body_width = 0


def url_to_safe_name(url):
    url = url.replace("http://", "").replace("https://", "").rstrip("/")
    return re.sub(r"[^\w-]", "_", url)


def get_matching_urls(html, base_url=None):
    soup = BeautifulSoup(html, "html.parser")
    urls = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if href.startswith("/") and base_url:
            href = base_url.rstrip("/") + href

        if re.match(pattern, href):
            urls.append(href)

    return urls

In [2]:
seed = "https://eecs.berkeley.edu"
max_pages = 100000

# --- Restart: rebuild visited + queue from already-saved HTML files ---
visited = set()
queue = deque()

print("Scanning existing html/ for already-crawled pages...")
for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue
    safe_name = filename[:-5]
    # Reverse safe_name back to a best-effort URL just for the visited key.
    # We store the safe_name→url mapping by re-reading the file for links instead.
    visited.add(safe_name)

# Re-read saved HTML to reconstruct the link frontier.
for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue
    with open(f"html/{filename}") as f:
        html = f.read()
    safe_name = filename[:-5]
    for u in get_matching_urls(html):
        if url_to_safe_name(u) not in visited:
            queue.append(u)

# If nothing was found (fresh start), begin from seed.
if not visited and not queue:
    queue.append(seed)

print(f"Resuming: {len(visited)} pages already crawled, {len(queue)} URLs queued.\n")
# ---------------------------------------------------------------------

while queue and len(visited) < max_pages:
    url = queue.popleft()
    safe_name = url_to_safe_name(url)

    if safe_name in visited:
        continue

    visited.add(safe_name)
    print(f"Visiting ({len(visited)}/{max_pages}): {url}")

    try:
        response = requests.get(url, timeout=5)
    except:
        continue

    if "Forbidden" in response.text:
        continue

    with open(f"html/{safe_name}.html", "w") as f:
        f.write(response.text)

    new_urls = get_matching_urls(response.text, base_url=url)
    for u in new_urls:
        if url_to_safe_name(u) not in visited:
            queue.append(u)

print(f"\nDone. Crawled {len(visited)} pages.")

Scanning existing html/ for already-crawled pages...
Resuming: 0 pages already crawled, 1 URLs queued.

Visiting (1/100000): https://eecs.berkeley.edu
Visiting (2/100000): https://eecs.berkeley.edu/resources/students/
Visiting (3/100000): https://eecs.berkeley.edu/resources/faculty-staff/
Visiting (4/100000): https://eecs.berkeley.edu/industry/
Visiting (5/100000): https://eecs.berkeley.edu/latest-news/
Visiting (6/100000): https://eecs.berkeley.edu/events/
Visiting (7/100000): https://eecs.berkeley.edu/connect/support/
Visiting (8/100000): https://eecs.berkeley.edu/about/
Visiting (9/100000): https://eecs.berkeley.edu/about/history/
Visiting (10/100000): https://eecs.berkeley.edu/about/history/first-women/
Visiting (11/100000): https://eecs.berkeley.edu/about/history/gier/
Visiting (12/100000): https://eecs.berkeley.edu/about/history/a-relatively-recent-history-women-doctoral-graduates-in-electrical-engineering-and-computer-sciences-1969-1981/
Visiting (13/100000): https://eecs.berkel

In [3]:
def clean_html_for_rag(html):
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup([
        "script", "style", "noscript",
        "nav", "footer", "header",
        "form", "input", "button", "select", "textarea",
        "svg", "img"
    ]):
        tag.decompose()

    noise_classes = [
        "sidebar", "ad", "ads", "promo",
        "cookie", "banner", "popup",
        "modal", "social", "share",
        "related", "footer"
    ]

    for tag in soup.find_all(True):
        classes = " ".join(tag.attrs.get("class", [])) if tag.attrs else ""
        if any(n in classes.lower() for n in noise_classes):
            tag.decompose()

    return h2t.handle(str(soup))


for filename in os.listdir("html"):
    if not filename.endswith(".html"):
        continue

    with open(f"html/{filename}") as f:
        html = f.read()

    text = clean_html_for_rag(html)

    out_name = filename[:-5]  # strip .html
    with open(f"cleaned_text/{out_name}.txt", "w") as f:
        f.write(text)

print(f"Done. Converted {len(os.listdir('html'))} files.")

Done. Converted 934 files.
